<a href="https://colab.research.google.com/github/alakebally/marketing-sales-roi-regression/blob/main/marketing_and_sales_data_evaluate_lr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Load Dataset**



In [ ]:
import pandas as pd

df = pd.read_csv("/content/marketing_and_sales_data_evaluate_lr.csv")
df.head()

**Explore dataset**

In [ ]:
df.info()

**Checking for missing values**

In [ ]:
df.isnull().sum()

**handle missing values by filling with mean  and clean dataset properly**

In [ ]:
df_clean = df.copy()

for col in df_clean.columns:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

**verify cleaning worked**

In [ ]:
df_clean.isnull().sum()

**Data cleaning explanation**
The dataset contained missing values across all variables, with TV having the highest number of missing entries. Since the dataset is numerical and suitable for regression analysis, missing values were handled using mean imputation to preserve data integrity without losing observations.

In [ ]:
df_clean.corr()

Understanding the Data

In [ ]:
df_clean.info()
df_clean.describe()

**Performing  exploratory data analysis (EDA) with visualizations**



*   Univariate Analysis
*   Bivariate Analysis





**Univariate Analysis**

Histogram (distribution of each variable)


In [ ]:
import matplotlib.pyplot as plt
df_clean.hist(figsize=(10,8))
plt.show()

**Box Plot**

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,6))
sns.boxplot(data=df_clean)
plt.show()

**Interpretation of Boxplots**
What This Boxplot Shows for the following variables


*   TV
*   Radio
*   Social media
*   sales


Each box shows:


*  Median (middle line)
*  Spread (IQR) (box height)
*  Outliers (dots)
*  Range (whiskers)


Interpretation (Variable by Variable)
1.   TV
Median ≈ 50
Wide spread (values roughly 10 → 100)

Meaning:

TV Advertising spend varies a lot across campaigns
Good variability → useful for regression
No extreme outliers → clean variable

2. Radio
Median ≈ 20
Smaller spread than TV
Has a few outliers (dots near ~45–50)

Meaning:

Radio budget is more consistent
A few campaigns spent unusually high → potential influence points


3. Social Media
Median ≈ 3–4
Very tight spread
A couple of small outliers (~10–12)
 Meaning:

Very low spend overall
Little variation → might be a weak predictor in regression


4. Sales (VERY IMPORTANT)
Median ≈ 190
Very wide spread (~30 → 300)

Meaning:

Sales vary a lot → good
No obvious extreme outliers → data looks usable

**Key Insights**

TV advertising spend  shows high variability, making it a strong candidate for predicting sales.

Radio  is moderately distributed but contains a few outliers that may influence the model.

Social media has very low variability, suggesting it may have limited impact on sales prediction.

Sales data is widely distributed, indicating sufficient variation for regression analysis.

**Bivariate Analysis**

In [ ]:
sns.scatterplot(x='TV', y='Sales', data=df_clean)
plt.show()

sns.scatterplot(x='Radio', y='Sales', data=df_clean)
plt.show()

sns.scatterplot(x='Social_Media', y='Sales', data=df_clean)
plt.show()

**Using the Correlation matrix to identify the Independent variable**


In [ ]:
corr = df_clean.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()

**Interpretation of the correlation matrix to Identify the Independent variable**
1. TV vs Sales → 1.00
Perfect relationship
TV alone explains 100% of Sales variation

 Conclusion:

TV is the BEST possible predictor(Independent variable)

2. Radio vs Sales → 0.87
Strong positive correlation

 Meaning:

Radio significantly affects Sales
But not as perfectly as TV

3. Social Media vs Sales → 0.53
Moderate correlation

Meaning:

Some relationship
But much weaker → not ideal for prediction


Final Conclusion
The correlation analysis shows that TV advertising as one of the marketing channels has a perfect positive correlation (r = 1.0) with sales, indicating a very strong linear relationship. Radio also demonstrates a strong correlation (r = 0.87), while social media shows a moderate relationship (r = 0.53). Based on these results, TV is selected as the independent variable for the regression model.

**Building an OLS Regression model using statmodels**

In [ ]:
#import library
import statsmodels.api as sm

In [ ]:
#Add constant
X = sm.add_constant(X)

In [ ]:
#define variables
X = df_clean[['TV']]   # independent variable
y = df_clean['Sales']  # dependent variable

In [ ]:
model = sm.OLS(y, X).fit()

**View Results**

In [ ]:
print(model.summary())

**Regression** **model**

Sales = 0.2849 + 3.5545 x TV

1. R-squared = 0.993

 This means:

99.3% of the variation in Sales is explained by TV advertising

  Interpretation:

The model has an excellent fit, indicating that TV advertising is a very strong predictor of sales.

2. TV Coefficient = 3.5545


Meaning:

For every 1 unit increase in TV advertising, Sales increase by
3.55 units

There is a strong positive relationship between TV advertising and sales, with sales increasing significantly as TV spending rises

3. Intercept (const) = 0.2849
Very small
P-value = 0.271 (> 0.05) → not significant

Meaning:

When TV = 0, Sales ≈ 0.28 (not very meaningful)

The intercept is not statistically significant, suggesting limited practical interpretation.

4. P-value (TV) = 0.000

Meaning:

The relationship between TV and Sales is statistically significant

5. F-statistic = 6.79e+05

 Very large → model is highly significant overall

 Final Conclusion

 The OLS regression model shows that TV advertising has a strong and statistically significant effect on sales. The model explains 99.3% of the variation in sales, indicating an excellent fit. The coefficient suggests that for every unit increase in TV advertising, sales increase by approximately 3.55 units. Therefore, TV advertising is the most effective predictor of sales among the variables considered

**Create diagnostic plots to test Linearity, Normality, and Homoscedasticity**

In [ ]:
#Get residual and fitted value
fitted_vals = model.fittedvalues
residuals = model.resid

**Test LInearity**


In [ ]:
#Use Residuals vs Fitted Plot

plt.figure(figsize=(6,4))
sns.scatterplot(x=fitted_vals, y=residuals)
plt.axhline(0, color='red')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted")
plt.show()

The residuals vs fitted plot shows that the majority of residuals are concentrated along a horizontal line around zero, indicating a strong linear relationship and good model fit. A small number of points display slight vertical and diagonal dispersion, suggesting minor variability and potential localized deviations. However, no clear systematic pattern or curvature is observed, and therefore the linearity assumption is considered to be satisfied.

**Test Normality of residuals**

(a) Histogram

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(residuals, kde=True)
plt.title("Histogram of Residuals")
plt.show()

The histogram of residuals shows a sharp peak around zero with the presence of extreme values in the tails. This indicates that the residuals are not normally distributed, exhibiting a leptokurtic(too many values near zero) distribution with heavy tails(extreme outliers).

**(b) Q-Q Plot**

In [ ]:
import statsmodels.api as sm

sm.qqplot(residuals, line='45')
plt.title("Q-Q Plot")
plt.show()

The Q–Q plot shows that the residuals deviate significantly from the 45-degree reference line, indicating that the assumption of normality is violated. The clustering of points around the center and the presence of extreme deviations at the tails suggest a leptokurtic distribution with heavy tails, likely caused by outliers.

**Test Homoscedascity**


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x=fitted_vals, y=residuals)
plt.axhline(0, color='red')
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted")
plt.show()

The residuals vs fitted plot shows that the residuals are randomly scattered around zero with no clear pattern, suggesting that the assumption of homoscedasticity is reasonably satisfied. However, the presence of a few extreme residuals indicates slight deviations from constant variance

 **Interpretation of  R-squared, coefficients, and p-values in business context**


1. **R-squared (0.993)**

it means 99.3% of the variation in Sales is explained by TV advertising



**Business interpretation:**

TV advertising is an extremely strong predictor of sales performance. Nearly all changes in sales can be explained by changes in TV advertising spend.





2. **Coefficients**

From the model:

Intercept (const) = 0.2849

TV coefficient = 3.5545

TV Coefficient (MOST important)

** What it means:**

*   For every 1 unit increase in TV advertising, Sales increase by ~3.55 units



**Business interpretation:**

Increasing TV advertising expenditure leads to a significant increase in sales. Specifically, each additional unit of TV advertising is associated with an average increase of about 3.55 units in sales



**Real meaning:**
TV advertising is a highly effective marketing channel, delivering strong ROI. Increasing the TV budget is likely to significantly boost revenue


**Intercept (0.2849)**

**What it means**

 Expected Sales when TV = 0

 **Business interpretation:**

When no money is spent on TV advertising, baseline sales are approximately 0.28 units

**p-value = 0.271** → NOT significant
So this baseline is not reliable

The intercept is not statistically significant, suggesting that baseline sales without TV advertising may not be meaningful in this model.

3.**P-values**

From the output:

 **TV p-value = 0.000**
**Intercept p-value = 0.271**

**TV Variable**
**What it means:**

**p-value: < 0.05** → statistically significant

** Business interpretation:**

TV advertising has a statistically significant impact on sales, meaning the relationship observed is highly unlikely to be due to chance.

 ** Translation**
This is a real effect, not random
You can confidently base decisions on it

 **Intercept**

  What it means:

 p-value=0.271 > 0.05 → NOT significant


 **Business interpretation:**

The intercept is not statistically significant, indicating that the estimated baseline level of sales (when TV advertising is zero) is not reliable.


**Final interpretation**

The regression model shows a very strong relationship between TV advertising and sales, with an R² of 0.993, indicating that 99.3% of the variation in sales is explained by TV advertising. The coefficient for TV advertising (3.5545) suggests that for every unit increase in TV advertising expenditure, sales increase by approximately 3.55 units, highlighting a strong positive impact. The p-value for TV advertising is less than 0.05, confirming that this relationship is statistically significant and not due to random chance. However, the intercept is not statistically significant, suggesting that the baseline level of sales in the absence of TV advertising is not reliable. Overall, the results indicate that TV advertising is a highly effective driver of sales



**ROI Based Recommendation**

**Core Insight from the  Model**



* Every 1 unit of TV advertising → ~3.55 units increase in Sales
*   Very strong relationship (R² ≈ 0.993)
*   Statistically significant (p < 0.05)


**What This Means for ROI**


ROI ≈ 3.55 : 1

 In plain terms:

For every 1naira  spent on TV advertising, the business generates approximately
3.55naira in sales.

 **Final Recommendation**

Based on the regression results, TV advertising demonstrates a strong return on investment, with each unit increase in advertising spend generating approximately 3.55 units in sales. This indicates that TV advertising is a highly effective driver of revenue. Therefore, it is recommended that the company allocate a larger portion of its marketing budget to TV advertising to maximize sales performance. However, budget increases should be implemented strategically and monitored over time to ensure that returns remain consistent and to avoid potential diminishing returns.